# 00c - Copernicus DEM 30m Elevation Downloader & Kaggle Packager

**Arab Youth Space Hackathon 2026** | Team: ليوا الخضراء | Theme: Precision Agriculture & Crop Intelligence

This notebook queries Microsoft Planetary Computer STAC for 30m Copernicus Digital Elevation Model (`cop-dem-glo-30`) tiles over Liwa Oasis and packages it as an isolated Kaggle Dataset (`assemelqersh/copernicus-dem-liwa`).

The mechanistic justification for this step is: Isolating DEM tile downloads ensures topographic slope/elevation features can be published as an independent, lightweight Kaggle dataset.

In [ ]:
# Automatic dependency check & installation for Kaggle/Colab/Local environments
# The mechanistic justification for this step is: Dynamically inspecting and installing required packages (planetary-computer, pystac-client, rasterio, geopandas, folium, h5py) prevents runtime ModuleNotFoundError exceptions across Kaggle runtime environments.
import subprocess
import sys

def install_if_missing(package, pip_name=None):
    if pip_name is None:
        pip_name = package
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

install_if_missing("planetary_computer", "planetary-computer")
install_if_missing("pystac_client", "pystac-client")
install_if_missing("pystac", "pystac")
install_if_missing("rasterio", "rasterio")
install_if_missing("geopandas", "geopandas")
install_if_missing("folium", "folium")
install_if_missing("h5py", "h5py")

# Step 1: Environment Setup & AOI Definition
# The mechanistic justification for this step is: Querying Copernicus GLO-30 DEM tiles over the Liwa Oasis bounding box provides elevation and slope covariates for groundwater drainage modeling.
import os
import sys
import json
import requests
import planetary_computer as pc
from pystac_client import Client

OUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "../data/processed/copernicus-dem-liwa"
os.makedirs(OUT_DIR, exist_ok=True)

LIWA_BBOX = [53.5, 22.8, 54.2, 23.3]
catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1", modifier=pc.sign_inplace)

dem_search = catalog.search(
    collections=["cop-dem-glo-30"],
    bbox=LIWA_BBOX,
    max_items=2
)

dem_items = list(dem_search.items())
print(f"Found {len(dem_items)} Copernicus DEM tiles over Liwa Oasis.")

for item in dem_items:
    if "data" in item.assets:
        dem_url = item.assets["data"].href
        save_dem = os.path.join(OUT_DIR, f"{item.id}.tif")
        if not os.path.exists(save_dem):
            r = requests.get(dem_url, stream=True)
            if r.status_code == 200:
                with open(save_dem, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
                mb = os.path.getsize(save_dem) / (1024 * 1024)
                print(f"Saved {save_dem} ({mb:.2f} MB)")

dem_meta = {
    "title": "Copernicus DEM 30m Liwa Oasis Elevation Dataset",
    "id": "assemelqersh/copernicus-dem-liwa",
    "licenses": [{"name": "CC-BY-4.0"}]
}
with open(os.path.join(OUT_DIR, "dataset-metadata.json"), "w") as f:
    json.dump(dem_meta, f, indent=2)

print(f"[SUCCESS] Copernicus DEM Dataset packaged cleanly in {OUT_DIR}")
